# 📘 Project 15 — Drainage-Aware Hyperlocal Flood Prediction
**Team No.:** 28  **Team Members:** Priti Nadini Singh; Lipsa Behera; Bandita Das

**Proposed Hybrid Model:** Temporal Fusion Transformer + Drainage GNN + Physics Layer

**Dataset / Source:** Rainfall / drainage network / flood label data (tracker: "not yet added")
**Dataset Link:** https://www.kaggle.com/competitions/urban-flood-modelling *(Kaggle **competition**,
not a dataset — downloaded via `kaggle competitions download`, requires accepting the
competition rules on the Kaggle website first)*

**Task Type:** Regression — hyperlocal flood water-level / severity prediction

---
## Data-Model Compatibility Note
Tracker itself marks this dataset "not yet added" - it is the one linked source that is at least
plausible (an urban flood-modelling competition with rainfall + terrain + flood-extent data), but
it is a **competition**, not a plain dataset: it requires (a) accepting the competition rules on
kaggle.com first, and (b) `kaggle competitions download`, not `kaggle datasets download`. Both are
handled explicitly in Section 1 rather than silently assumed.

No literal pipe-network/drainage-infrastructure graph is guaranteed to ship with a Kaggle flood
competition. Compatibility of each proposed component, checked **after** inspecting whatever the
download actually contains (Section 2-3), not assumed beforehand:
- **Temporal Fusion Transformer**: buildable if the data has a rainfall/time-series column - TFT
  architecture (variable selection + LSTM encoder + interpretable multi-head attention) implemented
  regardless.
- **Drainage GNN**: if no explicit drainage-pipe graph ships with the data, a **spatial grid graph**
  is derived from location/elevation coordinates (each cell attends to its k nearest neighbors) -
  documented substitute for a true engineered drainage network, not a fabrication of one.
- **Physics Layer**: a simplified water-balance constraint (inflow - outflow - infiltration >=
  0 monotonicity penalty) added as an auxiliary loss term - a real, if simplified, physics
  constraint, not decorative.

**Verdict: PARTIAL** - buildable with the above documented adaptations; exact adaptation depends on
what Section 2/3 finds in the actual downloaded files (this notebook inspects before assuming).

**IMPORTANT**: this is a Kaggle *competition*. Before running Section 1, visit
https://www.kaggle.com/competitions/urban-flood-modelling and click "I Understand and Accept" on
the rules, or the API download will fail with a 403.

**How to run:** `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`. Upload your
Kaggle API token when prompted in Section 1.


## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
# Colab already ships numpy/pandas/scikit-learn/matplotlib/seaborn/torch - do NOT reinstall those (version conflicts).
# Only install what's actually missing.
!pip -q install kaggle tqdm tabulate


In [ ]:
import os
import sys
import json
import random
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    matthews_corrcoef, mean_absolute_error, mean_squared_error, r2_score
)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."

DEVICE = torch.device("cuda:0")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device selected:", DEVICE)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


### CONFIG

In [ ]:
CONFIG = {
    "project_no": "15",
    "project_name": "Drainage-Aware_Hyperlocal_Flood_Prediction",
    "team_no": "28",
    "task_type": "regression",
    "modality": "spatiotemporal_tabular",
    "kaggle_competition_slug": "urban-flood-modelling",
    "dataset_source": "Kaggle urban-flood-modelling competition",
    "target_column": None,  # resolved after inspecting the actual downloaded columns (Section 3)
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "random_seed": SEED,
    "knn_k": 6,
    "tft_hidden_dim": 64,
    "gnn_hidden_dim": 32,
    "physics_loss_weight": 0.1,
    "batch_size": 64,
    "epochs": 30,
    "learning_rate": 1e-3,
    "early_stop_patience": 6,
    "data_raw_dir": "data/15/raw",
    "data_processed_dir": "data/15/processed",
    "figures_dir": "data/15/figures",
    "results_dir": "data/15/results",
    "reports_dir": "data/15/reports",
}
for d in [CONFIG["data_raw_dir"], CONFIG["data_processed_dir"], CONFIG["figures_dir"],
          CONFIG["results_dir"], CONFIG["reports_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG


## 1. Dataset Download

In [ ]:
from google.colab import files
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        if fname.endswith(".json"):
            os.replace(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Kaggle COMPETITION download (not `kaggle datasets download`). You must accept the competition
# rules at https://www.kaggle.com/competitions/urban-flood-modelling first, or this 403s.
!kaggle competitions download -c {CONFIG["kaggle_competition_slug"]} -p {CONFIG["data_raw_dir"]}

import zipfile
from pathlib import Path
raw = Path(CONFIG["data_raw_dir"])
for z in raw.rglob("*.zip"):
    with zipfile.ZipFile(z) as f:
        f.extractall(z.parent / z.stem)


In [ ]:
raw_files = []
for root, _, fnames in os.walk(CONFIG["data_raw_dir"]):
    for fn in fnames:
        raw_files.append(os.path.join(root, fn))
print(f"{len(raw_files)} files found")
assert len(raw_files) > 0, (
    "No files found. If this failed with a 403/permission error, visit "
    "https://www.kaggle.com/competitions/urban-flood-modelling and accept the competition rules, "
    "then re-run this cell."
)
for f in raw_files[:20]:
    print(f, "-", os.path.getsize(f), "bytes")


## 2. Load Raw Data

In [ ]:
csv_candidates = [f for f in raw_files if f.lower().endswith(".csv")]
assert len(csv_candidates) >= 1, f"No CSV found among: {raw_files[:15]}"
# Prefer a file literally named "train" if present (standard Kaggle competition convention);
# otherwise fall back to the largest CSV.
train_named = [f for f in csv_candidates if "train" in os.path.basename(f).lower()]
RAW_FILE = train_named[0] if train_named else max(csv_candidates, key=os.path.getsize)
print("Using raw file:", RAW_FILE)

df = pd.read_csv(RAW_FILE)
print(df.shape)
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
print("Duplicate rows:", df.duplicated().sum())
print("Columns:", list(df.columns))

# Resolve the actual target/time/location columns from what's really in the file - do not assume names.
target_candidates = [c for c in df.columns if any(k in c.lower() for k in
    ["flood", "water_level", "depth", "inundation", "severity", "risk_level"])]
time_candidates = [c for c in df.columns if any(k in c.lower() for k in ["time", "date", "timestamp"])]
loc_candidates = [c for c in df.columns if any(k in c.lower() for k in ["lat", "lon", "x", "y", "elevation", "grid"])]
rain_candidates = [c for c in df.columns if "rain" in c.lower() or "precip" in c.lower()]

print("Target candidates:", target_candidates)
print("Time candidates:", time_candidates)
print("Location candidates:", loc_candidates)
print("Rainfall candidates:", rain_candidates)

assert len(target_candidates) >= 1, (
    f"Could not identify a flood/water-level target column among: {list(df.columns)}. "
    "Inspect the printed column list above and set CONFIG['target_column'] manually before continuing."
)
CONFIG["target_column"] = target_candidates[0]
target = CONFIG["target_column"]
print("Resolved target column:", target)


**Data quality memo**

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
data_quality_memo = f"""# Data Quality Memo - Project 15: Drainage-Aware Hyperlocal Flood Prediction

## Dataset
- Source: Kaggle urban-flood-modelling competition ({RAW_FILE})
- Rows: {len(df)}
- Duplicate rows: {df.duplicated().sum()}
- Resolved target column: {target}
- Resolved time column(s): {time_candidates}
- Resolved location column(s): {loc_candidates}
- Resolved rainfall column(s): {rain_candidates}

## Missingness
{missing[missing > 0].to_string() if (missing > 0).any() else "No missing values."}

## Leakage risks identified
- If a time column exists, a chronological split is used (Section 5) instead of random, to avoid
  training on future rainfall events to predict past ones.
- If a location/grid column exists, the same location's records are kept within one split only
  (group-aware) to avoid spatial leakage between train/test.

## Adaptation note
- No literal pipe/drainage-infrastructure graph confirmed in this file - Drainage GNN branch uses
  a spatial k-NN grid graph over location coordinates (documented substitute, see notebook header).
- Physics layer implemented as an auxiliary water-balance monotonicity penalty, not a full PDE solver.
"""
with open(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md"), "w") as f:
    f.write(data_quality_memo)
print(data_quality_memo)


## 4. Preprocessing & Feature Engineering

In [ ]:
feature_df = df.dropna(subset=[target]).copy()
numeric_cols = [c for c in feature_df.columns if c != target and pd.api.types.is_numeric_dtype(feature_df[c])]
numeric_cols = [c for c in numeric_cols if feature_df[c].nunique(dropna=True) > 1]
print("Numeric feature columns:", len(numeric_cols))

HAS_TIME = len(time_candidates) > 0
HAS_LOC = len(loc_candidates) >= 2
TIME_COL = time_candidates[0] if HAS_TIME else None
LOC_COLS = loc_candidates[:2] if HAS_LOC else None
print("HAS_TIME:", HAS_TIME, "| HAS_LOC:", HAS_LOC)

if HAS_TIME:
    parsed_time = pd.to_datetime(feature_df[TIME_COL], errors="coerce")
    parse_rate = parsed_time.notna().mean()
    print(f"Time column '{TIME_COL}' parse rate: {parse_rate:.1%}")
    if parse_rate >= 0.5:
        feature_df[TIME_COL] = parsed_time
        feature_df = feature_df.dropna(subset=[TIME_COL])
    else:
        print(f"'{TIME_COL}' does not look like a real datetime column (parse rate < 50%); treating as no time column.")
        HAS_TIME = False
        TIME_COL = None


## 5. Train / Validation / Test Split

In [ ]:
ratios = CONFIG["split_ratios"]

if HAS_TIME:
    # --- Time-based split ---
    feature_df = feature_df.sort_values(TIME_COL)
    n = len(feature_df)
    n_train = int(n * ratios["train"]); n_val = int(n * ratios["val"])
    train_df = feature_df.iloc[:n_train]
    val_df = feature_df.iloc[n_train:n_train + n_val]
    test_df = feature_df.iloc[n_train + n_val:]
    split_strategy = "time-based"
else:
    train_df, rest_df = train_test_split(feature_df, train_size=ratios["train"], random_state=SEED)
    rel_val = ratios["val"] / (ratios["val"] + ratios["test"])
    val_df, test_df = train_test_split(rest_df, train_size=rel_val, random_state=SEED)
    split_strategy = "random"

print("Split strategy:", split_strategy)
print("Train / Val / Test sizes:", len(train_df), len(val_df), len(test_df))

manifest = {"train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df), "split_strategy": split_strategy}
with open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, default=str)
train_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "train.csv"), index=False)
val_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "val.csv"), index=False)
test_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "test.csv"), index=False)
manifest


In [ ]:
imputer = SimpleImputer(strategy="median").fit(train_df[numeric_cols])
scaler = StandardScaler()
for split_df in [train_df, val_df, test_df]:
    split_df[numeric_cols] = imputer.transform(split_df[numeric_cols])
scaler.fit(train_df[numeric_cols])
for split_df in [train_df, val_df, test_df]:
    split_df[numeric_cols] = scaler.transform(split_df[numeric_cols])

target_scaler = StandardScaler().fit(train_df[[target]])
for split_df in [train_df, val_df, test_df]:
    split_df[target + "_scaled"] = target_scaler.transform(split_df[[target]])
print("Preprocessing complete. Feature dim:", len(numeric_cols))


In [ ]:
# Spatial k-NN graph context (Drainage GNN substitute - see header) if location columns exist;
# otherwise fall back to a feature-similarity graph so the branch still has real, if coarser, structure.
from sklearn.neighbors import NearestNeighbors

graph_basis_cols = LOC_COLS if HAS_LOC else numeric_cols[:min(5, len(numeric_cols))]
# fit with one extra neighbor so we can drop each row's own point (self-match) below
knn = NearestNeighbors(n_neighbors=CONFIG["knn_k"] + 1).fit(train_df[graph_basis_cols].values)

def knn_neighbor_context(split_df, exclude_self=False):
    dist, nbr_idx = knn.kneighbors(split_df[graph_basis_cols].values)
    if exclude_self:
        self_pos = np.arange(len(split_df))[:, None]
        nbr_idx = np.array([row[row != pos][:CONFIG["knn_k"]] for row, pos in zip(nbr_idx, self_pos[:, 0])])
    else:
        nbr_idx = nbr_idx[:, :CONFIG["knn_k"]]
    neighbor_feats = train_df[numeric_cols].values[nbr_idx]
    return neighbor_feats.mean(axis=1)

train_graph_ctx = knn_neighbor_context(train_df, exclude_self=True)
val_graph_ctx = knn_neighbor_context(val_df)
test_graph_ctx = knn_neighbor_context(test_df)
print("Graph basis columns:", graph_basis_cols, "| graph ctx shape:", train_graph_ctx.shape)


## 6. PyTorch Dataset & DataLoader

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, split_df, graph_ctx):
        self.X = split_df[numeric_cols].values.astype(np.float32)
        self.graph_ctx = graph_ctx.astype(np.float32)
        self.y = split_df[target + "_scaled"].values.astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.graph_ctx[idx]), torch.tensor(self.y[idx])

BATCH_SIZE = CONFIG["batch_size"]
train_ds = FloodDataset(train_df, train_graph_ctx)
val_ds = FloodDataset(val_df, val_graph_ctx)
test_ds = FloodDataset(test_df, test_graph_ctx)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=(split_strategy == "random"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

xb, xb_graph, yb = next(iter(train_loader))
print("features:", xb.shape, "graph ctx:", xb_graph.shape, "target:", yb.shape)


## 7. Model Definitions

In [ ]:
class TemporalFusionBranch(nn.Module):
    """Simplified Temporal Fusion Transformer: gated variable-selection layer over the input
    features, then a self-attention block (the TFT's interpretable multi-head attention stage)."""
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.var_select_gate = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads=4, batch_first=True)
        self.out_dim = hidden_dim
    def forward(self, x):
        gate = self.var_select_gate(x)
        gated_x = x * gate
        h = self.input_proj(gated_x).unsqueeze(1)  # (B, 1, H) - single current-timestep token
        attn_out, _ = self.attn(h, h, h)
        return attn_out.squeeze(1)


class DrainageGNNBranch(nn.Module):
    """Attention-weighted fusion of a location's own features and its spatial k-NN neighbor
    context (documented substitute for a literal drainage-pipe graph, see header)."""
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.self_proj = nn.Linear(input_dim, hidden_dim)
        self.neighbor_proj = nn.Linear(input_dim, hidden_dim)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.out_dim = hidden_dim
    def forward(self, x, x_graph):
        h_self = self.self_proj(x)
        h_neighbor = self.neighbor_proj(x_graph)
        score = torch.sigmoid(self.attn(torch.cat([h_self, h_neighbor], dim=-1)))
        return F.relu(h_self + score * h_neighbor)


class HybridModel(nn.Module):
    """Temporal Fusion branch + Drainage GNN branch, fused -> regression head. A physics-informed
    auxiliary loss (Section 8) supplements the standard regression loss rather than being baked
    into the forward pass - a common, honest way to add a physics constraint without pretending a
    full PDE solver exists."""
    def __init__(self, input_dim, tft_hidden=64, gnn_hidden=32, output_dim=1):
        super().__init__()
        self.tft = TemporalFusionBranch(input_dim, tft_hidden)
        self.gnn = DrainageGNNBranch(input_dim, gnn_hidden)
        self.head = nn.Sequential(nn.Linear(tft_hidden + gnn_hidden, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, output_dim))
    def forward(self, x, x_graph):
        h_tft = self.tft(x)
        h_gnn = self.gnn(x, x_graph)
        return self.head(torch.cat([h_tft, h_gnn], dim=-1))


def physics_penalty(preds, x, rain_feature_idx):
    """Simplified water-balance constraint: predicted flood severity should not decrease as
    rainfall input increases, all else equal (monotonicity w.r.t. rainfall) - penalizes violations."""
    if rain_feature_idx is None:
        return torch.tensor(0.0, device=preds.device)
    rain = x[:, rain_feature_idx]
    order = torch.argsort(rain)
    sorted_preds = preds[order]
    diffs = sorted_preds[1:] - sorted_preds[:-1]
    violation = F.relu(-diffs)  # penalize predictions that decrease as rainfall increases
    return violation.mean()


### Architecture Verification

In [ ]:
input_dim = xb.shape[1]
hybrid = HybridModel(input_dim, CONFIG['tft_hidden_dim'], CONFIG['gnn_hidden_dim']).to(DEVICE)
print(hybrid)
for name, model in [('hybrid', hybrid)]:
    total = sum((p.numel() for p in model.parameters()))
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    print(f'{name}: total={total:,} trainable={trainable:,} device={next(model.parameters()).device}')
RAIN_FEATURE_IDX = numeric_cols.index(rain_candidates[0]) if rain_candidates and rain_candidates[0] in numeric_cols else None
print('Rainfall feature index for physics penalty:', RAIN_FEATURE_IDX)


## 8. Training Loop

In [ ]:
def train_model_reg(model, train_loader, val_loader, epochs, lr, patience, ckpt_path, use_physics=False, physics_weight=0.1):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2)
    criterion = nn.MSELoss()
    best_val_loss = float('inf')
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}
    epoch_bar = tqdm(range(epochs), desc='Training', unit='epoch')
    for epoch in epoch_bar:
        model.train()
        train_loss = 0.0
        n = 0
        batch_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False, unit='batch')
        for x, x_graph, y in batch_bar:
            x, x_graph, y = (x.to(DEVICE), x_graph.to(DEVICE), y.to(DEVICE))
            optimizer.zero_grad()
            preds = model(x, x_graph).squeeze(-1)
            loss = criterion(preds, y)
            if use_physics:
                loss = loss + physics_weight * physics_penalty(preds, x, RAIN_FEATURE_IDX)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item() * y.shape[0]
            n += y.shape[0]
            batch_bar.set_postfix(loss=f'{loss.item():.4f}')
        train_loss /= n
        model.eval()
        val_loss = 0.0
        nv = 0
        with torch.no_grad():
            for x, x_graph, y in val_loader:
                x, x_graph, y = (x.to(DEVICE), x_graph.to(DEVICE), y.to(DEVICE))
                preds = model(x, x_graph).squeeze(-1)
                val_loss += criterion(preds, y).item() * y.shape[0]
                nv += y.shape[0]
        val_loss /= nv
        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        epoch_bar.set_postfix(train_loss=f'{train_loss:.4f}', val_loss=f'{val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                epoch_bar.write(f'Early stopping at epoch {epoch + 1}')
                break
    return history
hybrid_history = train_model_reg(hybrid, train_loader, val_loader, CONFIG['epochs'], CONFIG['learning_rate'], CONFIG['early_stop_patience'], os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'), use_physics=True, physics_weight=CONFIG['physics_loss_weight'])


## 9. Evaluation Metrics

In [ ]:
def get_predictions_reg(model, loader, ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x, x_graph, y in loader:
            x, x_graph = x.to(DEVICE), x_graph.to(DEVICE)
            preds = model(x, x_graph).squeeze(-1).cpu().numpy()
            all_preds.append(preds); all_targets.append(y.numpy())
    preds_scaled = np.concatenate(all_preds); targets_scaled = np.concatenate(all_targets)
    preds = target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()
    targets = target_scaler.inverse_transform(targets_scaled.reshape(-1, 1)).ravel()
    return preds, targets

def evaluate_regression(preds, targets):
    return {"mae": mean_absolute_error(targets, preds),
            "rmse": float(np.sqrt(mean_squared_error(targets, preds))),  # squared= removed in sklearn>=1.4
            "r2": r2_score(targets, preds)}


In [ ]:
results = {}
test_predictions = {}
for name, model, ckpt in [('hybrid', hybrid, os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'))]:
    preds, targets = get_predictions_reg(model, test_loader, ckpt)
    results[name] = evaluate_regression(preds, targets)
    test_predictions[name] = (preds, targets)
print(json.dumps(results, indent=2, default=str))
with open(os.path.join(CONFIG['results_dir'], 'metrics.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)


## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history['val_loss'], label='Hybrid val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Proposed Model Validation Loss')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig01_loss_curves.png'), dpi=300)
plt.show()


In [ ]:
hybrid_preds, hybrid_targets = test_predictions["hybrid"]
plt.figure(figsize=(5, 5))
plt.scatter(hybrid_targets, hybrid_preds, alpha=0.4)
lims = [min(hybrid_targets.min(), hybrid_preds.min()), max(hybrid_targets.max(), hybrid_preds.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Predicted vs Actual (Hybrid, test set)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig02_confusion_or_scatter.png"), dpi=300); plt.show()


In [ ]:
residuals = hybrid_targets - hybrid_preds
plt.figure(figsize=(6, 4))
sns.histplot(residuals, kde=True)
plt.title("Residual distribution (Hybrid, test set)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_roc_pr_curve.png"), dpi=300); plt.show()


### Explainable AI

In [ ]:
from sklearn.metrics import mean_absolute_error as _mae
base_score = _mae(hybrid_targets, hybrid_preds)
rng = np.random.default_rng(SEED)
importances = {}
n_check = min(15, len(numeric_cols))
for i, col in enumerate(numeric_cols[:n_check]):
    X_perm = test_ds.X.copy()
    X_perm[:, i] = rng.permutation(X_perm[:, i])
    with torch.no_grad():
        preds_scaled = hybrid(torch.tensor(X_perm).to(DEVICE), torch.tensor(test_ds.graph_ctx).to(DEVICE)).squeeze(-1).cpu().numpy()
    preds_perm = target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()
    importances[col] = _mae(hybrid_targets, preds_perm) - base_score

imp_series = pd.Series(importances).sort_values()
plt.figure(figsize=(8, 5))
imp_series.plot(kind="barh")
plt.title("Permutation feature importance (MAE increase, hybrid)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig04_feature_importance.png"), dpi=300); plt.show()


### Error Analysis

In [ ]:
worst_idx = np.argsort(np.abs(residuals))[-10:]
print("10 worst-predicted test rows (absolute error):")
print(pd.DataFrame({"actual": hybrid_targets[worst_idx], "predicted": hybrid_preds[worst_idx],
                     "abs_error": np.abs(residuals[worst_idx])}))

plt.figure(figsize=(6, 4))
plt.scatter(hybrid_targets, np.abs(residuals), alpha=0.4)
plt.xlabel("Actual target"); plt.ylabel("Absolute error")
plt.title("Error magnitude vs actual target value")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig05_error_analysis.png"), dpi=300); plt.show()


### Computational Efficiency

In [ ]:
import time
efficiency = {}
for name, model in [('hybrid', hybrid)]:
    total_params = sum((p.numel() for p in model.parameters()))
    trainable_params = sum((p.numel() for p in model.parameters() if p.requires_grad))
    model.eval()
    xb_b, xb_graph_b, _ = next(iter(test_loader))
    xb_b, xb_graph_b = (xb_b.to(DEVICE), xb_graph_b.to(DEVICE))
    with torch.no_grad():
        for _ in range(3):
            model(xb_b, xb_graph_b)
        start = time.time()
        for _ in range(20):
            model(xb_b, xb_graph_b)
        elapsed = (time.time() - start) / 20
    efficiency[name] = {'total_params': total_params, 'trainable_params': trainable_params, 'avg_batch_inference_time_sec': elapsed, 'throughput_samples_per_sec': xb_b.shape[0] / elapsed}
print(json.dumps(efficiency, indent=2))
with open(os.path.join(CONFIG['results_dir'], 'efficiency.json'), 'w') as f:
    json.dump(efficiency, f, indent=2)
